# Assignment 2 — Clustering Student Data with Unsupervised Learning (K-Means)
**Dataset:** UCI Student Performance (`student-por.csv`)

**Aim:** Apply K-Means clustering to demographic, academic, social and lifestyle features (excluding G1/G2/G3 to avoid leakage), select K via elbow + silhouette, and profile/visualize the clusters.


## 1. Library imports

In [1]:
import io, os
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.makedirs("outputs", exist_ok=True)
print("random_state =", RANDOM_STATE)

random_state = 42


## 2. Dataset loading (official UCI source)

In [2]:
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"
raw = urllib.request.urlopen(UCI_URL, timeout=60).read()
zf = zipfile.ZipFile(io.BytesIO(raw))
with zf.open("student-por.csv") as f:
    df = pd.read_csv(f, sep=";")
print("Shape:", df.shape)
print("Columns:", list(df.columns))

Shape: (649, 33)
Columns: ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']


## 3. Dataset preview

In [3]:
df.head()


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [4]:
print(df.describe(include="all").T.head(35))
print("\nMissing values total:", int(df.isnull().sum().sum()))

            count unique     top freq       mean       std   min   25%   50%  \
school        649      2      GP  423        NaN       NaN   NaN   NaN   NaN   
sex           649      2       F  383        NaN       NaN   NaN   NaN   NaN   
age         649.0    NaN     NaN  NaN  16.744222  1.218138  15.0  16.0  17.0   
address       649      2       U  452        NaN       NaN   NaN   NaN   NaN   
famsize       649      2     GT3  457        NaN       NaN   NaN   NaN   NaN   
Pstatus       649      2       T  569        NaN       NaN   NaN   NaN   NaN   
Medu        649.0    NaN     NaN  NaN   2.514638  1.134552   0.0   2.0   2.0   
Fedu        649.0    NaN     NaN  NaN   2.306626  1.099931   0.0   1.0   2.0   
Mjob          649      5   other  258        NaN       NaN   NaN   NaN   NaN   
Fjob          649      5   other  367        NaN       NaN   NaN   NaN   NaN   
reason        649      4  course  285        NaN       NaN   NaN   NaN   NaN   
guardian      649      3  mother  455   

## 4. Data preprocessing — ColumnTransformer + Pipeline (exclude G1/G2/G3)

In [5]:
LEAK = ["G1", "G2", "G3"]
X_raw = df.drop(columns=LEAK)
print(f"Clustering features: {X_raw.shape[1]} columns (excluded {LEAK})")
cat_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_raw.select_dtypes(exclude=["object"]).columns.tolist()
print("Numeric:", num_cols)
print("Categorical:", cat_cols)

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])
X = preprocess.fit_transform(X_raw)
print("Encoded+scaled matrix:", X.shape)
print("Feature names:", len(preprocess.get_feature_names_out()))

Clustering features: 30 columns (excluded ['G1', 'G2', 'G3'])
Numeric: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']
Categorical: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']


Encoded+scaled matrix: (649, 56)
Feature names: 56


C:\Users\ns479\AppData\Local\Temp\ipykernel_19840\2953766900.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()


## 5. K-Means over K=2..10 — Elbow (inertia) + Silhouette

In [6]:
ks = list(range(2, 11))
inertias, sils = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    lab = km.fit_predict(X)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X, lab))
    print(f"k={k:2d} inertia={km.inertia_:10.2f} silhouette={sils[-1]:.4f}")
res = pd.DataFrame({"k": ks, "inertia": inertias, "silhouette": sils})
res.to_csv("outputs/k_metrics.csv", index=False)
best_k = int(res.loc[res["silhouette"].idxmax(), "k"])
print(f"\nBest K by silhouette: {best_k}")

k= 2 inertia=  11988.82 silhouette=0.1119


k= 3 inertia=  11221.13 silhouette=0.0679
k= 4 inertia=  10785.32 silhouette=0.0691


k= 5 inertia=  10504.77 silhouette=0.0562


k= 6 inertia=  10244.86 silhouette=0.0579


k= 7 inertia=  10022.14 silhouette=0.0578


k= 8 inertia=   9822.45 silhouette=0.0522


k= 9 inertia=   9654.71 silhouette=0.0472


k=10 inertia=   9502.13 silhouette=0.0475

Best K by silhouette: 2


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(ks, inertias, marker="o")
axes[0].set_xlabel("K"); axes[0].set_ylabel("Inertia"); axes[0].set_title("Elbow method")
axes[1].bar(ks, sils)
axes[1].set_xlabel("K"); axes[1].set_ylabel("Silhouette score"); axes[1].set_title("Silhouette scores")
plt.suptitle("K-Means model selection")
plt.tight_layout(); plt.savefig("outputs/elbow_silhouette.png", dpi=150); plt.close()
print("Saved: outputs/elbow_silhouette.png")

Saved: outputs/elbow_silhouette.png


## 6. Final K-Means model (best K by silhouette)

In [8]:
kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE)
labels = kmeans.fit_predict(X)
df["cluster"] = labels
print(f"Final K={best_k}, silhouette={silhouette_score(X, labels):.4f}, inertia={kmeans.inertia_:.2f}")
sizes = df["cluster"].value_counts().sort_index()
print("\nCluster sizes:"); print(sizes)
print("\nProportions:"); print((sizes / len(df)).round(4))
sizes.to_csv("outputs/cluster_sizes.csv", header=["count"])

Final K=2, silhouette=0.1119, inertia=11988.82

Cluster sizes:
cluster
0    463
1    186
Name: count, dtype: int64

Proportions:
cluster
0    0.7134
1    0.2866
Name: count, dtype: float64


## 7. Cluster profiles (feature means + G3 held-out mean)

In [9]:
profile_num = df.groupby("cluster")[num_cols + ["G3"]].mean().round(3)
print("Numeric means per cluster (incl. held-out G3):")
print(profile_num)
profile_num.to_csv("outputs/cluster_profiles_numeric.csv")
prof_cat = pd.crosstab(df["cluster"], df["sex"], normalize="index").round(3)
print("\nSex distribution per cluster:"); print(prof_cat)
print("\nAddress distribution per cluster:")
print(pd.crosstab(df["cluster"], df["address"], normalize="index").round(3))

Numeric means per cluster (incl. held-out G3):
            age   Medu   Fedu  traveltime  studytime  failures  famrel  \
cluster                                                                  
0        16.583  2.626  2.356       1.473      2.082     0.104   3.937   
1        17.145  2.237  2.183       1.806      1.554     0.516   3.914   

         freetime  goout   Dalc   Walc  health  absences      G3  
cluster                                                           
0           2.991  2.840  1.140  1.747   3.404     2.911  12.603  
1           3.651  4.043  2.403  3.608   3.866     5.522  10.172  

Sex distribution per cluster:
sex          F      M
cluster              
0        0.695  0.305
1        0.328  0.672

Address distribution per cluster:
address      R      U
cluster              
0        0.259  0.741
1        0.414  0.586


## 8. Two-dimensional cluster visualization (PCA 2-D)

In [10]:
pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
X2 = pca2.fit_transform(X)
print(f"PCA-2 explained variance ratio: {pca2.explained_variance_ratio_.round(4)} (total {pca2.explained_variance_ratio_.sum():.4f})")
plt.figure(figsize=(8, 6))
for c in sorted(df["cluster"].unique()):
    m = df["cluster"].to_numpy() == c
    plt.scatter(X2[m, 0], X2[m, 1], s=22, alpha=0.7, label=f"Cluster {c} (n={(m).sum()})")
plt.xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]:.1%})")
plt.ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]:.1%})")
plt.title(f"K-Means clusters (K={best_k}) in PCA 2-D space")
plt.legend(); plt.tight_layout()
plt.savefig("outputs/clusters_pca2d.png", dpi=150); plt.close()
print("Saved: outputs/clusters_pca2d.png")

PCA-2 explained variance ratio: [0.1198 0.1046] (total 0.2244)


Saved: outputs/clusters_pca2d.png


## 9. Result and conclusion
- Preprocessing used a `ColumnTransformer` (StandardScaler for numerics, OneHotEncoder for categoricals); G1/G2/G3 excluded from clustering inputs.
- Elbow + silhouette tables/plots saved to `outputs/k_metrics.csv` and `outputs/elbow_silhouette.png`; best-K chosen by maximum silhouette.
- Cluster sizes, numeric profiles (with held-out G3 mean) and PCA 2-D visualization saved to `outputs/`.
- All metric values are computed at runtime (no invented numbers).
